In [ ]:
# ── Cell 1: Install ────────────────────────────────────────────────────────────
!pip install -q transformers accelerate bitsandbytes datasets \
    sentence-transformers faiss-cpu rouge-score nltk bert-score \
    matplotlib seaborn scipy scikit-learn

import nltk; nltk.download('punkt', quiet=True)

In [ ]:
# ── Cell 2: Mount Drive & Paths ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, pickle, numpy as np, random
random.seed(42); np.random.seed(42)

In [ ]:
# ── Cell 3: Load SQuAD — guaranteed relevant passages in corpus ────────────────
from datasets import load_dataset

print("Loading SQuAD validation set...")
squad = load_dataset("squad", split="validation")

# Build corpus: unique passages
corpus, seen = [], set()
for item in squad:
    key = item['context'][:100]
    if key not in seen:
        seen.add(key)
        corpus.append({'id': f'ctx_{len(corpus)}',
                       'title': item['title'],
                       'text': item['context']})

print(f"✅ Corpus: {len(corpus)} unique passages")

# Build queries (first 500), track gold passage index
title_ctx_to_id = {}
for i, p in enumerate(corpus):
    title_ctx_to_id[(p['title'], p['text'][:100])] = i

queries = []
for item in squad:
    if len(queries) >= 500: break
    gold_id = title_ctx_to_id.get((item['title'], item['context'][:100]), -1)
    queries.append({'id': item['id'],
                    'question': item['question'],
                    'answers': item['answers']['text'],
                    'gold_corpus_id': gold_id,
                    'title': item['title']})

print(f"✅ Queries: {len(queries)} | With gold passage: {sum(1 for q in queries if q['gold_corpus_id']>=0)}")

with open(f'{BASE}/corpus.pkl', 'wb') as f: pickle.dump(corpus, f)
with open(f'{BASE}/queries.pkl', 'wb') as f: pickle.dump(queries, f)
print("✅ Corpus + queries saved")


In [ ]:
# ── Cell 4: Build FAISS Index ──────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer
import faiss

print("Loading sentence-transformers (all-MiniLM-L6-v2)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("Encoding corpus (~3 min)...")
texts = [f"{p['title']}. {p['text']}" for p in corpus]
embs = embedder.encode(texts, batch_size=128, show_progress_bar=True,
                        convert_to_numpy=True).astype('float32')
faiss.normalize_L2(embs)

idx = faiss.IndexFlatIP(embs.shape[1])
idx.add(embs)
print(f"✅ FAISS index: {idx.ntotal} passages, dim={embs.shape[1]}")

faiss.write_index(idx, f'{BASE}/faiss_index.bin')
np.save(f'{BASE}/corpus_embeddings.npy', embs)
print("✅ Index saved.\n\n👉 SETUP DONE — Open Notebook 1 with GPU runtime")